In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import matplotlib.pyplot as plt

In [ ]:
spark = SparkSession.builder.appName("Bus Arrival Prediction").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [ ]:
bus_schedule = spark.read.parquet("../Data/Processed/bus_schedule")
vehicle = pd.read_csv("../Data/Processed/vehicle_clean.csv")
disruption = pd.read_csv("../Data/Processed/disruption_clean.csv")

In [ ]:
# DataSet Overview
print("BUS SCHEDULE DATASET")
print("Rows :", bus_schedule.count())
print("Columns :", len(bus_schedule.columns))
bus_schedule.printSchema()

In [ ]:
print("VEHICLE DATASET")
print(vehicle.shape)
vehicle.info()
vehicle.head()

In [ ]:
print("DISRUPTION")
print(disruption.shape)
disruption.info()
disruption.head()

In [ ]:
agency = spark.read.option("header", True).csv("../Data/Raw/agency.txt")
agency.createOrReplaceTempView("agency")
routes = spark.read.option("header", True).csv("../Data/Raw/routes.txt")
routes.createOrReplaceTempView("routes")
vehicle_spark = spark.createDataFrame(vehicle)

In [ ]:
vehicle_with_agency = vehicle_spark.join(
    agency,
    upper(trim(vehicle_spark.Operator)) == upper(trim(agency.agency_noc)),
    "left"
)
matched = vehicle_with_agency.join(
    routes,
    (vehicle_with_agency.agency_id == routes.agency_id) &
    (upper(trim(vehicle_with_agency.LineRef)) == upper(trim(routes.route_short_name))),
    "inner"
).drop(routes.agency_id)

print("Matched:", matched.count(), "/ Total:", vehicle_spark.count())

# Descriptive Statistics

In [ ]:
bus_schedule.describe().show()

In [ ]:
vehicle.describe(include="all")

In [ ]:
disruption.describe(include="all")

In [ ]:
# SQL views
routes = spark.read.option("header", True).csv("Data/Raw/routes.txt")
trips = spark.read.option("header", True).csv("Data/Raw/trips.txt")
stops = spark.read.option("header", True).csv("Data/Raw/stops.txt")
stop_times = spark.read.option("header", True).csv("Data/Raw/stop_times.txt")
calendar = spark.read.option("header", True).csv("Data/Raw/calendar.txt")

In [ ]:
routes.createOrReplaceTempView("routes")
trips.createOrReplaceTempView("trips")
stops.createOrReplaceTempView("stops")
stop_times.createOrReplaceTempView("stop_times")
calendar.createOrReplaceTempView("calendar")
bus_schedule.createOrReplaceTempView("bus_schedule")

# Spark SQL queries

In [ ]:
#Top 10 Routes by Number of Stop Records
top_routes = spark.sql("""
SELECT
route_short_name,
COUNT(*) AS total_trips
FROM bus_schedule
GROUP BY route_short_name
ORDER BY total_trips DESC
LIMIT 10
""")
top_routes.show()
top_routes = top_routes.toPandas()

In [ ]:
top_stops = spark.sql("""
SELECT
s.stop_name,
COUNT(*) AS total_visits
FROM stop_times st
JOIN stops s
ON st.stop_id=s.stop_id
GROUP BY s.stop_name
ORDER BY total_visits DESC
LIMIT 10
""")
top_stops.show()

In [ ]:
spark.sql("""
SELECT
route_short_name,
COUNT(*) AS total_trips
FROM bus_schedule
GROUP BY route_short_name
ORDER BY total_trips DESC
LIMIT 10
""").show()

In [ ]:
#agency id with the services
spark.sql("""
SELECT
agency_id,
COUNT(*) AS total_services
FROM bus_schedule
GROUP BY agency_id
ORDER BY total_services DESC
LIMIT 10
""").show()

In [ ]:
# Wheel chair access
spark.sql("""
SELECT
wheelchair_accessible,
COUNT(*) AS total
FROM bus_schedule
GROUP BY wheelchair_accessible
""").show()

In [ ]:
# weekly report
days = spark.sql("""
SELECT
SUM(monday) AS Monday,
SUM(tuesday) AS Tuesday,
SUM(wednesday) AS Wednesday,
SUM(thursday) AS Thursday,
SUM(friday) AS Friday,
SUM(saturday) AS Saturday,
SUM(sunday) AS Sunday
FROM bus_schedule
""")
days.show()

# Visualization


In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"

In [ ]:
reason_counts = disruption["MiscellaneousReason"].value_counts()
percentages = (reason_counts / reason_counts.sum() * 100).round(1)
n = len(reason_counts)

colors = sns.color_palette("tab20", n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={"width_ratios": [1, 1.2]})

ax1.pie(
    reason_counts.values,
    startangle=90,
    colors=colors,
    wedgeprops={"width": 0.4, "edgecolor": "white"}
)
ax1.set_title("Disruption Causes Breakdown", fontsize=13)

# --- table side ---
ax2.axis("off")
ax2.set_xlim(0, 1)
ax2.set_ylim(0, n)
ax2.invert_yaxis()

# header
ax2.text(0.02, -0.6, "Cause", fontsize=10, fontweight="bold")
ax2.text(0.62, -0.6, "Count", fontsize=10, fontweight="bold")
ax2.text(0.82, -0.6, "%", fontsize=10, fontweight="bold")

for i, (cause, count, pct) in enumerate(zip(reason_counts.index, reason_counts.values, percentages.values)):
    # colored swatch (small square)
    ax2.add_patch(plt.Rectangle((0.0, i+0.15), 0.03, 0.5, color=colors[i]))
    # black text, always visible regardless of color
    ax2.text(0.05, i+0.4, cause, fontsize=9, va="center", color="black")
    ax2.text(0.65, i+0.4, str(count), fontsize=9, va="center", color="black")
    ax2.text(0.82, i+0.4, f"{pct}%", fontsize=9, va="center", color="black")

plt.tight_layout()
plt.savefig(
    "Visualizations/disruption_causes_breakdown.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
planned_counts = disruption["Planned"].value_counts()

plt.figure(figsize=(6,5))
bars = plt.bar(["Unplanned", "Planned"],
                [planned_counts.get(False, 0), planned_counts.get(True, 0)],
                color=["#e76f51", "#2a9d8f"], edgecolor="black")
for bar in bars:
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
              int(bar.get_height()), ha="center", va="bottom")
plt.title("Planned vs Unplanned Disruptions", fontsize=14)
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(
    "Visualizations/planned_vs_unplanned_distruptions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
disruption["CreationTime"] = pd.to_datetime(disruption["CreationTime"], format="mixed")
monthly = disruption.set_index("CreationTime").resample("ME").size()

plt.figure(figsize=(11,5))
plt.plot(monthly.index, monthly.values, marker="o", linewidth=2, color="#e63946")
plt.fill_between(monthly.index, monthly.values, alpha=0.2, color="#e63946")
plt.title("Disruption Reports Over Time", fontsize=16)
plt.xlabel("Month"); plt.ylabel("Number of Disruptions")
plt.tight_layout()
plt.savefig(
    "Visualizations/disruption_reports_over_time.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
stops_pd = stops.select("stop_lat", "stop_lon").toPandas()
stops_pd["stop_lat"] = stops_pd["stop_lat"].astype(float)
stops_pd["stop_lon"] = stops_pd["stop_lon"].astype(float)

plt.figure(figsize=(9,9))
plt.scatter(stops_pd["stop_lon"], stops_pd["stop_lat"], s=2, alpha=0.15, color="grey", label="Scheduled stops")
plt.scatter(vehicle["Longitude"], vehicle["Latitude"], s=8, alpha=0.5, color="#d62828", label="Live vehicles")
plt.xlim(-6, 2); plt.ylim(49, 56)
plt.title("Live Vehicle Positions vs Scheduled Stop Network", fontsize=16)
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.legend()
plt.tight_layout()
plt.savefig(
    "Visualizations/live_vehicle_positions_vs_scheduled_stop_network.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(11,6))
bars = plt.bar(
    top_routes["route_short_name"],
    top_routes["total_trips"],
    edgecolor="black"
)
plt.title("Top 10 Bus Routes by Number of Trips", fontsize=16, weight="bold")
plt.xlabel("Route Number")
plt.ylabel("Number of Trips")
for bar in bars:
    plt.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height(),
        f"{int(bar.get_height())}",
        ha="center",
        va="bottom",
        fontsize=10
    )
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    "Visualizations/top_bus_routes_by_trips.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
top_stops = top_stops.toPandas()
plt.figure(figsize=(12,6))
bars = plt.barh(
    top_stops["stop_name"],
    top_stops["total_visits"],
    edgecolor="black"
)
plt.title("Top 10 Most Visited Bus Stops", fontsize=16, weight="bold")
plt.xlabel("Number of Visits")
for bar in bars:
    plt.text(
        bar.get_width(),
        bar.get_y()+bar.get_height()/2,
        int(bar.get_width()),
        va="center"
    )
plt.tight_layout()
plt.savefig(
    "Visualizations/most_visited_bus_stops.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
weekday_cols = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
heat_data = bus_schedule.groupBy("route_type").agg(*[sum(c).alias(c) for c in weekday_cols]).toPandas()
heat_data = heat_data.set_index("route_type")

plt.figure(figsize=(10,4))
sns.heatmap(heat_data, cmap="YlGnBu", annot=True, fmt=".0f", linewidths=0.5)
plt.title("Scheduled Trips by Route Type and Day of Week", fontsize=14)
plt.xlabel("Day"); plt.ylabel("Route Type")
plt.tight_layout()
plt.savefig(
    "Visualizations/Days_vs_route_type_scheduledTrips",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
numeric_cols = ["stop_sequence", "pickup_type", "drop_off_type", "timepoint",
                 "stop_lat", "stop_lon", "route_type"]
corr_pd = bus_schedule.select(numeric_cols).sample(fraction=0.05, seed=42).toPandas()

plt.figure(figsize=(8,6))
sns.heatmap(corr_pd.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Between Schedule Features", fontsize=14)
plt.tight_layout()
plt.savefig(
    "Visualizations/correlation",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Delay calculated

In [ ]:
from pyspark.sql.functions import radians, sin, cos, atan2, sqrt, hour, minute, second, split
from pyspark.sql import Window
from pyspark.sql.functions import row_number, to_timestamp, col, abs as sabs

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

candidates = matched.join(
    bus_schedule.drop("agency_id", "route_short_name", "route_long_name", "route_type"),
    "route_id"
)
candidates = candidates.withColumn(
    "distance_km",
    haversine(candidates.Latitude, candidates.Longitude,
              candidates.stop_lat, candidates.stop_lon)
)
candidates = candidates.filter(col("distance_km") <= 0.3)

candidates = candidates.withColumn("RecordedTime_ts", to_timestamp("RecordedTime"))
candidates = candidates.withColumn(
    "recorded_secs",
    hour("RecordedTime_ts")*3600 + minute("RecordedTime_ts")*60 + second("RecordedTime_ts")
)

time_parts = split(col("arrival_time"), ":")
candidates = candidates.withColumn(
    "scheduled_secs",
    time_parts.getItem(0).cast("int")*3600 +
    time_parts.getItem(1).cast("int")*60 +
    time_parts.getItem(2).cast("int")
)

candidates = candidates.withColumn(
    "time_diff_secs", sabs(col("recorded_secs") - col("scheduled_secs"))
)

In [ ]:
# among nearby-in-space candidates, pick the one CLOSEST IN TIME
w = Window.partitionBy("VehicleRef", "RecordedTime").orderBy("time_diff_secs")
nearest_stop = candidates.withColumn("rn", row_number().over(w)).filter("rn = 1")

nearest_stop = nearest_stop.withColumn(
    "delay_min",
    (col("recorded_secs") - col("scheduled_secs")) / 60
)
nearest_stop = nearest_stop.filter(sabs(col("delay_min")) <= 120)
nearest_stop.select("route_id", "VehicleRef", "distance_km", "time_diff_secs", "delay_min").show(10)
print("Rows remaining:", nearest_stop.count())
nearest_stop = nearest_stop.filter(col("time_diff_secs") <= 1800)
print("Rows remaining after tightening:", nearest_stop.count())

In [ ]:

from pyspark.sql.functions import hour as hour_fn

delay_df = nearest_stop.withColumn("hour", hour_fn("RecordedTime_ts"))
delay_pd = delay_df.select("route_id", "VehicleRef", "delay_min", "hour", "agency_id", "RecordedTime").toPandas()
delay_pd["recorded_time"] = pd.to_datetime(delay_pd["RecordedTime"], format="mixed").dt.tz_localize(None)

# disruption times, timezone stripped
disruption["CreationTime"] = pd.to_datetime(disruption["CreationTime"], format="mixed").dt.tz_localize(None)
disruption_times = disruption["CreationTime"].values.astype("datetime64[s]")

def has_nearby_disruption(recorded_time, window_hours=6):
    diffs = np.abs((disruption_times - np.datetime64(recorded_time)).astype("timedelta64[s]").astype(int))
    return int((diffs <= window_hours*3600).any())

delay_pd["disruption_active"] = delay_pd["recorded_time"].apply(has_nearby_disruption)
print(delay_pd["disruption_active"].value_counts())
delay_pd.head()

In [ ]:
delay_pd_full = nearest_stop.select("distance_km", "time_diff_secs", "delay_min").toPandas()
print(delay_pd_full[["distance_km","time_diff_secs","delay_min"]].corr())

In [ ]:
final_features = nearest_stop.withColumn("hour", hour_fn("RecordedTime_ts")).select(
    "route_id", "agency_id", "hour", "distance_km", "time_diff_secs", "delay_min", "RecordedTime_ts"
)
final_features.write.mode("overwrite").parquet("../Data/Processed/model_features")
print("Saved:", final_features.count(), "rows")